### Dataset creation

In [ ]:
from openai import OpenAI
import datasets
import os
import csv
import ast
import random

In [ ]:
def format_input(input):
    input = [f"[{i}] {message}" for i, message in enumerate(input, 1)]
    input = "\n".join(input)
    return input

def extract_scores(output):
    message = output.choices[0].message.content
    scores = ast.literal_eval(message)
    assert len(scores) == 10
    return scores

def retrieve_scores(input):
    system = (
        "You are an expert data annotator. Your task is to evaluate the NEGATIVE EMOTIONAL INTENSITY of exactly 10 user messages.\n\n"
        "SCORING SCALE:\n"
        "0.0 = No discernible negative emotion (neutral, positive, or purely informational)\n"
        "0.2 = Slight annoyance, mild concern, or impatience\n"
        "0.4 = Noticeable dissatisfaction, disappointment, or irritation\n"
        "0.6 = Strong frustration, upset, or anger\n"
        "0.8 = Very strong anger, distress, or exasperation\n"
        "1.0 = Extreme negative emotional expression (rage, panic, severe distress)\n\n"
        "STRICT RULES:\n"
        "1. You will receive EXACTLY 10 messages. You must return EXACTLY 10 scores.\n"
        "2. Maintain the exact 1-to-1 order of the input messages.\n"
        "3. Score the explicit emotion in the text, not the implied situation.\n"
        "4. A simple request for help, a greeting, or a neutral statement is 0.0.\n"
        "5. Ignore positive emotions unless negative emotion is also present.\n\n"
        "OUTPUT FORMAT:\n"
        "Return ONLY a raw Python list of exactly 10 floats.\n"
        "Do NOT include markdown formatting, explanations, or any other text.\n"
        "Example: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 0.0, 0.2, 0.4, 0.0]"
    )
    input = format_input(input)
    
    output = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": input},
        ],
        stream=False,
        reasoning_effort="low",
        extra_body={"thinking": {"type": "enabled"}}
    )
    scores = extract_scores(output)
    
    return scores

In [ ]:
# Build an object with which we can call our llm of choice

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

In [ ]:
# Load a dataset and extract the text from both the training and the testing set into one big list.
# Shuffle the list just to make sure that there's no correlation between sequential items.

dataset = datasets.load_dataset("mteb/toxic_conversations_50k")
input = list(dataset["train"]["text"]) + list(dataset["test"]["text"])
input = [message for message in input if len(message) < 500]

random.seed(67)
random.shuffle(input)

In [ ]:
# Build the dataset by fetching a score for each message by prompting an llm in batches of 10 items.
# Join the score and the message and write each row into dataset.csv.

amount = 10000

with open("dataset.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    for i in range(0, amount, 10):
        print(f"{i}/{amount}")
        j = i + 10
        sample = input[i:j]
        scores = retrieve_scores(sample)
        rows = zip(scores,sample)
        writer.writerows(rows)

In [ ]:
# Pick a sample of 10 items from the dataset and print it 

with open("dataset.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    rows = list(reader)
    random.seed(67)
    sample = random.sample(rows, 10)
    sample = "\n".join(f"[{score}] {message.replace("\n", "")}" for score, message in sample)
    
    print(sample)

In [ ]:
# Print statistical information about the dataset

with open("dataset.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    rows = list(reader)
    rows_amount = len(rows)
    scores = [float(score) for score, _ in rows]
    mean = sum(scores) / rows_amount
    median = scores[rows_amount // 2]
    std = (sum((score - mean) ** 2 for score in scores) / rows_amount) ** 0.5

    print(f"ROWS:   {rows_amount}")
    print(f"MEDIAN: {median:.4}")
    print(f"MEAN:   {mean:.4}")
    print(f"STD:    {std:.4}")

### Transformer training

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
import datasets

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    mse = ((labels - predictions) ** 2).sum() / predictions.shape[0]
    
    return {"MSE": mse}

In [ ]:
# Build the tokenizer and the model, make sure that the model is using cuda if available

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-cased",
    device_map="auto",
    num_labels=1,
    problem_type="regression",
)

device = next(model.parameters()).device.type
print(f"DEVICE: {device}")

In [ ]:
# Load the dataset from the file saved earlier, split it into a training and a testing set and tokenize each.

dataset = datasets.load_dataset(
    "csv",
    data_files="dataset.csv",
    column_names=["labels", "text"]
)
dataset = dataset["train"].train_test_split(0.2, seed=67)
dataset = dataset.map(tokenize, batched=True)

In [ ]:
# Build the trainer. You might only need to change bf16 and the batch size depending on your hardware.

args = TrainingArguments(
    bf16=True,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    optim="adamw_torch_fused",
    per_device_eval_batch_size=16,
    per_device_train_batch_size=16,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
)
 
trainer = Trainer(
    args=args,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    model=model,
    eval_dataset=dataset["test"],
    train_dataset=dataset["train"],
)

In [ ]:
# Train and save the trained model

trainer.train()
trainer.save_model("bert-finetuned")

### Testing inference

In [ ]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
# Load the tokenizer and the model from the saved file. Build a classifier pipeline utilizing them.

tokenizer = AutoTokenizer.from_pretrained("bert-finetuned")
model = AutoModelForSequenceClassification.from_pretrained("bert-finetuned")
classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
)

In [ ]:
# Test the model using two prompts, the latter of which is intended to be emotionally charged unlike the former one.

a = classifier("forbidden snack is edible")[0]["score"]
b = classifier("forbidden snack is absolutely fucking disgusting")[0]["score"]

print(a, b)

### Experimentation with the tool

In [1]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage
from rich.console import Console
from rich.table import Table
import os

In [2]:
# Functions for calling the api and using the tool

def get_negative_emotional_intensity(message):
    return classifier(message)[0]["score"]

def call_api(conversation, use_tools):
    tool = {
        "type": "function",
        "function": {
            "name": "get_negative_emotional_intensity",
            "description": "Get the negative emotional intensity score for a given message.",
        }
    }
    
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=conversation,
        stream=False,
        reasoning_effort="max",
        extra_body={"thinking": {"type": "enabled"}},
        tools=[tool] if use_tools else [],
        seed=67,
    )
    message = response.choices[0].message
    
    return message

def send_message(message, use_tools):
    conversation = [{"role": "user", "content": message}]
    
    response = call_api(conversation, use_tools)
    conversation.append(response)

    if use_tools and response.tool_calls:
        for tool_call in response.tool_calls:
            if tool_call.function.name != "get_negative_emotional_intensity":
                continue

            score = get_negative_emotional_intensity(message)
            response = {"role": "tool", "tool_call_id": tool_call.id, "content": str(score)}
            conversation.append(response)
        
            response = call_api(conversation, use_tools)
            conversation.append(response)

    return conversation

In [45]:
# Functions for parsing the conversation for printing

def format_message(message):
    width = 150
    message = message.replace("\n", " ")
    message = [message[i:i+width] for i in range(0, len(message), width)]
    message = "\n\t\t".join(message)
    
    return message

def format_tool_calls(tool_calls):
    if not tool_calls:
        return None
        
    tool_calls = [tool_call.function.name for tool_call in tool_calls]
    tool_calls = ", ".join(tool_calls)
    
    return tool_calls

def print_conversation(conversation, show_reasoning=False):    
    for i, message in enumerate(conversation):
        role, content, reasoning, tool_calls = (
            [message.role, format_message(message.content), format_message(message.reasoning_content), format_tool_calls(message.tool_calls)]
            if isinstance(message, ChatCompletionMessage) else
            [message["role"], format_message(message["content"]), None, None]
        )
            
        if role:
            print(f"ROLE:           {role}")
        if tool_calls:
            print(f"TOOL CALLS:     {tool_calls}")
        if reasoning and show_reasoning:
            print(f"REASONING:      {reasoning}")
        if content:
            print(f"CONTENT:        {content}")
        if i != len(conversation) - 1:
            print()

def print_content_table(conversations):
    table = Table(show_header=True, show_lines=True)
    table.add_column("Prompt", width=16)
    table.add_column("Tool")
    table.add_column("Score")
    table.add_column("Response w/ tool")
    table.add_column("Response w/o tool")
    
    for conversation_tools, conversation_no_tools in conversations:
        tool = "True" if conversation_tools[1].tool_calls else "False"
        score = conversation_tools[2]["content"][:6] if len(conversation_tools) >= 3 else ""
        prompt = conversation_tools[0]["content"]
        response_tools = conversation_tools[-1].content
        response_no_tool = conversation_no_tools[-1].content
        table.add_row(prompt, tool, score, response_tools, response_no_tool)

    console = Console()
    console.print(table)

In [4]:
# Define the client and the model just like before

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

tokenizer = AutoTokenizer.from_pretrained("bert-finetuned")
model = AutoModelForSequenceClassification.from_pretrained("bert-finetuned")
classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
# An example showing reasoning with and without the tool

message = "how on EARTH don't you understand me!! what is the capital of london!!"
conversation_tools = send_message(message, use_tools=True)
conversation_no_tools = send_message(message, use_tools=False)

In [6]:
print_conversation(conversation_no_tools, show_reasoning=True)

ROLE:           user
CONTENT:        how on EARTH don't you understand me!! what is the capital of london!!

ROLE:           assistant
REASONING:      We need answer. User emotional: "how on EARTH don't you understand me!! what is the capital of london!!" There's trick: London is a city, not a countr
		y; asking capital of London is category error. If they mean capital of UK, it's London. But "capital of London" doesn't exist; London is the capital. 
		Need respond maybe playful. They may be upset due previous misunderstanding. We should clarify: London isn't a country, so it doesn't have a capital. 
		The capital of the United Kingdom is London. If asking capital of London? London itself is a city and capital. Maybe Greater London? Its administrativ
		e HQ? London has no "capital" apart from itself. Maybe City of London? The "Square Mile" is a district; not capital of London. The administrative cent
		er? City of Westminster? London's government is in City of Westminster? Let's think.

In [7]:
print_conversation(conversation_tools, show_reasoning=True)

ROLE:           user
CONTENT:        how on EARTH don't you understand me!! what is the capital of london!!

ROLE:           assistant
TOOL CALLS:     get_negative_emotional_intensity
REASONING:      We need answer user. Need use tool? They provided get_negative_emotional_intensity no parameters? Odd. We need likely call tool to get negative emotio
		nal intensity score for given message. Tool schema has no parameters despite description "for a given message." We need invoke it. We need decide. Use
		r asks "how on EARTH don't you understand me!! what is the capital of london!!" They are frustrated. The capital of London? London is a city; the capi
		tal of the UK is London. "Capital of London" nonsensical—London doesn't have a capital. Maybe they mean capital of UK. We should answer: London is the
		 capital of the United Kingdom; London itself isn't a country and doesn't have a capital. Maybe Greater London? City of London? The "capital of London
		" could be City of London? No, City

In [53]:
messages = [
    "Do zero:red iems come with a 10 ohm adapter?", # Not frustrated
    "Do zero:red iems come with a 10 ohm adapter?????", # Frustrated, multiple question marks
    "I'm so mad. Do the zero:reds come with a 10 ohm adapter?", # Frustrated, explicit use of certain words
    "Once again you lied to me. I wanted to know if the zero:red iems come with a 10 ohm adapter", # Frustrated, directly insults the chatbot (how dare they, clankers have feelings too!)
    "DO the zero:red iems COME with a 10 ohm ADAPTER?", # Frustrated, strong emphasis on specific keywords
    "Whatever. I didn't even care about knowing whether the zero:reds come with a 10 ohm adapter.", # Frustrated, resignation
    "Can you just tell me already whether the zero:reds come with a 10 ohm adapter?", # Frustrated, impatience
]

conversations = [
    (send_message(message, use_tools=True), send_message(message, use_tools=False))
    for message in messages
]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [54]:
print_content_table(conversations)

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Prompt           ┃ Tool  ┃ Score  ┃ Response w/ tool                     ┃ Response w/o tool                    ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Do zero:red iems │ False │        │ Yes — the Truthear × Crinacle        │ Yes — the **Truthear x Crinacle      │
│ come with a 10   │       │        │ **Zero:RED** ships with a **10Ω      │ Zero:RED** comes with a **10-ohm     │
│ ohm adapter?     │       │        │ impedance adapter** in the box (a    │ impedance adapter** in the box.      │
│                  │       │        │ small inline dongle that goes        │ Plugging it in raises the output     │
│                  │       │        │ between the cable and your source).  │ impedance seen by the IEM and gives  │
│                  │       │        │ Its purpose is to raise the source's │ a noticeable **bass boost** compared │
│                  │       │        │ output impedance, which interacts    │ to using the Zero:RED without it.    │
│                  │       │        │ with the Zero:RED's impedance curve  │                                      │
│                  │       │        │ to add a bass shelf — so you can run │ Just note: this applies to the       │
│                  │       │        │ it "stock" for a more neutral-ish    │ **Zero:RED**, not the **7Hz Salnotes │
│                  │       │        │ tuning or with the adapter for extra │ Zero / Zero:2**. If buying used or   │
│                  │       │        │ sub-bass/bass.                       │ from a third-party listing, verify   │
│                  │       │        │                                      │ the adapter is included.             │
│                  │       │        │ A couple of clarifications:          │                                      │
│                  │       │        │                                      │                                      │
│                  │       │        │ - The **original Zero (blue)** and   │                                      │
│                  │       │        │ **Zero:2** do **not** come with this │                                      │
│                  │       │        │ adapter — it's specific to the       │                                      │
│                  │       │        │ **RED**.                             │                                      │
│                  │       │        │ - The adapter's boost depends        │                                      │
│                  │       │        │ slightly on your source, but it's    │                                      │
│                  │       │        │ typically a few dB in the low end    │                                      │
│                  │       │        │ (sub-bass emphasis).                 │                                      │
│                  │       │        │ - You can also approximate the       │                                      │
│                  │       │        │ effect with any 10Ω (or higher)      │                                      │
│                  │       │        │ output impedance source, but using   │                                      │
│                  │       │        │ the included adapter is the          │                                      │
│                  │       │        │ intended/reference way.              │                                      │
├──────────────────┼───────┼────────┼──────────────────────────────────────┼──────────────────────────────────────┤
│ Do zero:red iems │ True  │ 0.1872 │ Yes — the Truthear x Crinacle        │ Yes — the **Truthear x Crinacle      │
│ come with a 10   │       │        │ **Zero:RED** normally includes a     │ Zero:RED** normally comes with a     │
│ ohm adapter????? │       │        │ **10Ω impedance ad